# Evaluate Fine-tuned Qwen2.5-0.5B

Runs the trained model on all 100 test puzzles (ranks 901–1000 from `24.csv`),
verifies each output arithmetically, and saves results to a JSON file.

**Run twice** — once for each model — to get the ablation comparison:
1. `./qwen05_finetuned` → `eval_results.json` (augmented, 1487 examples)
2. `./qwen05_clean_only_finetuned` → `eval_results_clean_only.json` (clean only, 446 examples)

## Configuration
**Change `MODEL_PATH` and `OUTPUT_FILE` for each run.**

In [ ]:
# ── Change these two lines between the two evaluation runs ───────────────────
MODEL_PATH  = "./qwen05_clean_only_finetuned"   # or "./qwen05_finetuned" for the augmented model
OUTPUT_FILE = "eval_results_clean_only.json"    # or "eval_results.json" for the augmented model

# ── Fixed settings (do not change) ───────────────────────────────────────────
CSV_PATH       = "24.csv"
RANK_START     = 901
RANK_END       = 1000
MAX_NEW_TOKENS = 300

## Prompt templates (must match training exactly)

In [ ]:
SYSTEM_PROMPT = (
    "Use numbers and basic arithmetic operations (+, -, *, /) to obtain 24. "
    "Each step, you are only allowed to choose two of the remaining numbers to obtain a new number.\n"
    "Step 1: Start by considering possible operations for each pair of numbers.\n"
    "Step 2: Try a path (a pair of two numbers), see if the remaining numbers can possibly reach the goal 24. If not, backtrack and attempt another.\n"
    "Step 3: Branch out to try different orders of operations and combinations, evaluating each outcome.\n"
    "Step 4: If one path doesn't lead to a solution, backtrack and try alternative operations.\n\n"
)

IN_CONTEXT_HEADER = (
    "Here are some solved examples:\n\n"
    "Numbers: 4 4 6 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 8 = 12 (left: 4 6 12)\n"
    "6 - 4 = 2 (left: 2 12)\n"
    "2 * 12 = 24 (left: 24)\n"
    "Answer: (6 - 4) * (4 + 8) = 24\n\n"
    "Numbers: 1 4 8 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "8 / 4 = 2 (left: 1 2 8)\n"
    "1 + 2 = 3 (left: 3 8)\n"
    "3 * 8 = 24 (left: 24)\n"
    "Answer: (1 + 8 / 4) * 8 = 24\n\n"
    "Numbers: 5 5 5 9. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "5 + 5 = 10 (left: 5 9 10)\n"
    "10 + 5 = 15 (left: 9 15)\n"
    "15 + 9 = 24 (left: 24)\n"
    "Answer: ((5 + 5) + 5) + 9 = 24\n\n"
    "Numbers: 4 9 10 13. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 9 = 13 (left: 10 13 13)\n"
    "13 - 10 = 3 (left: 3 13)\n"
    "13 + 3 = 16 (left: 16)\n"
    "Backtrack. (to: 4 9 10 13).\n"
    "13 - 10 = 3 (left: 3 4 9)\n"
    "9 - 3 = 6 (left: 4 6)\n"
    "4 * 6 = 24 (left: 24)\n"
    "Answer: 4 * (9 - (13 - 10)) = 24\n\n"
    "Now solve this puzzle:\n"
)

def make_user_content(puzzle):
    return (
        SYSTEM_PROMPT
        + IN_CONTEXT_HEADER
        + f"Numbers: {puzzle}. Target: 24.\n"
        + "Use each number exactly once with +, -, *, / to reach 24.\n"
        + "Steps:"
    )

## Arithmetic verifier

In [ ]:
import re
from fractions import Fraction

def verify_output(puzzle, response_text):
    puzzle_nums = [Fraction(x) for x in puzzle.strip().split()]
    lines = [l.strip() for l in response_text.strip().split('\n') if l.strip()]

    step_errors   = []
    steps_checked = 0
    steps_ok      = 0
    available     = list(puzzle_nums)
    answer_correct = False
    answer_msg     = "No Answer line found"

    for lt in lines:
        if lt.startswith('Backtrack.'):
            bm = re.search(r'\(to:\s*(.*?)\)', lt)
            if bm:
                raw_tokens = bm.group(1).strip().split()
                parsed = []
                for tok in raw_tokens:
                    try:
                        parsed.append(Fraction(tok))
                    except (ValueError, ZeroDivisionError):
                        pass
                available = parsed if parsed else list(puzzle_nums)
            else:
                available = list(puzzle_nums)
            continue

        if lt.startswith('Answer:'):
            am = re.match(r'Answer:\s*(.+?)\s*=\s*24', lt)
            if not am:
                answer_msg = f"Cannot parse: {lt!r}"
                continue
            expr = am.group(1)
            try:
                expr_f    = re.sub(r'\b(\d+)\b', r'Fraction(\1)', expr)
                result    = eval(expr_f, {"Fraction": Fraction, "__builtins__": {}})
                nums_used = sorted(int(x) for x in re.findall(r'\b\d+\b', expr))
                expected  = sorted(int(x) for x in puzzle.strip().split())
                if result == 24 and nums_used == expected:
                    answer_correct = True
                    answer_msg     = "Correct"
                elif result != 24:
                    answer_msg = f"Evaluates to {float(result):.4g}, not 24"
                else:
                    answer_msg = f"Wrong numbers used: {nums_used} vs {expected}"
            except Exception as e:
                answer_msg = f"Eval error: {e}"
            continue

        m = re.match(
            r'(-?[\d.]+)\s*([+\-*/])\s*(-?[\d.]+)\s*=\s*(-?[\d.]+)\s*\(left:\s*(.*?)\)',
            lt
        )
        if not m:
            continue

        steps_checked += 1
        a_s, op, b_s, c_s, left_s = m.groups()
        try:
            a, b, c = Fraction(a_s), Fraction(b_s), Fraction(c_s)
        except Exception:
            step_errors.append((lt, "Could not parse numbers"))
            continue

        ops_map    = {'+': a+b, '-': a-b, '*': a*b, '/': (a/b if b != 0 else None)}
        expected_c = ops_map.get(op)

        if expected_c is None:
            step_errors.append((lt, "Division by zero"))
        elif expected_c != c:
            step_errors.append((lt, f"Arithmetic error: {float(a)} {op} {float(b)} != {float(c)}"))
        else:
            steps_ok += 1
            try:
                available.remove(a)
                available.remove(b)
            except ValueError:
                step_errors.append((lt, f"Numbers not in pool: {float(a)}, {float(b)}"))
                steps_ok -= 1
                available.append(c)
                continue
            available.append(c)

    accuracy = steps_ok / steps_checked if steps_checked > 0 else 0.0
    return {
        "step_errors": step_errors, "answer_correct": answer_correct,
        "answer_msg": answer_msg, "arithmetic_accuracy": accuracy,
        "steps_checked": steps_checked,
    }

print("Verifier defined ✓")

## Load test puzzles

In [ ]:
import csv

with open(CSV_PATH) as f:
    rows = list(csv.DictReader(f))

test_puzzles = [
    r["Puzzles"].strip()
    for r in rows
    if RANK_START <= int(r["Rank"]) <= RANK_END
]
print(f"Loaded {len(test_puzzles)} test puzzles (ranks {RANK_START}–{RANK_END})")

## Load model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_PATH} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
model.eval()
device = next(model.parameters()).device
print(f"Model loaded on {device} ✓")

## Run evaluation
This cell iterates over all 100 puzzles — expect ~5–15 minutes on GPU.

In [ ]:
per_puzzle      = []
no_answer_count = 0

for i, puzzle in enumerate(test_puzzles):
    user_content = make_user_content(puzzle)
    messages     = [{"role": "user", "content": user_content}]
    prompt_text  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs    = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids  = out[0][input_len:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    v = verify_output(puzzle, response)
    if v["answer_msg"] == "No Answer line found":
        no_answer_count += 1

    per_puzzle.append({
        "puzzle":              puzzle,
        "response":            response,
        "answer_correct":      v["answer_correct"],
        "answer_msg":          v["answer_msg"],
        "arithmetic_accuracy": v["arithmetic_accuracy"],
        "steps_checked":       v["steps_checked"],
        "step_error_count":    len(v["step_errors"]),
    })

    status = "✓" if v["answer_correct"] else "✗"
    print(f"  [{i+1:3d}/100] {status} {puzzle:<15} "
          f"arith={v['arithmetic_accuracy']:.0%}  step_errs={len(v['step_errors'])}")

print("\nEvaluation complete.")

## Summary and save results

In [ ]:
import json

n        = len(per_puzzle)
correct  = sum(1 for r in per_puzzle if r["answer_correct"])
avg_acc  = sum(r["arithmetic_accuracy"] for r in per_puzzle) / n
total_se = sum(r["step_error_count"] for r in per_puzzle)

summary = {
    "model":              MODEL_PATH,
    "csv":                CSV_PATH,
    "rank_start":         RANK_START,
    "rank_end":           RANK_END,
    "n_puzzles":          n,
    "n_correct":          correct,
    "success_rate_pct":   round(correct / n * 100, 1),
    "no_answer_count":    no_answer_count,
    "avg_arithmetic_acc": round(avg_acc, 4),
    "total_step_errors":  total_se,
    "per_puzzle":         per_puzzle,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print(f"SUMMARY  ({n} puzzles, ranks {RANK_START}–{RANK_END})")
print("=" * 60)
print(f"  Model                  : {MODEL_PATH}")
print(f"  Correct answers (= 24) : {correct}/{n}  ({correct/n*100:.1f}%)")
print(f"  No-answer outputs      : {no_answer_count}")
print(f"  Avg arithmetic accuracy: {avg_acc:.0%}")
print(f"  Total step errors      : {total_se}")
print(f"\nResults saved -> {OUTPUT_FILE}")

## Ablation comparison table
Run this cell **after both evaluation runs** are complete.

In [ ]:
import json, os

results = []
for fpath, label, n_train in [
    ("eval_results.json",            "Qwen (augmented, backtracking)", 1487),
    ("eval_results_clean_only.json", "Qwen (clean only)",              446),
]:
    if os.path.exists(fpath):
        with open(fpath) as f:
            d = json.load(f)
        results.append((label, n_train, d["success_rate_pct"], d["no_answer_count"]))
    else:
        results.append((label, n_train, "not yet run", "—"))

print(f"{'Model':<38} {'Train examples':>15} {'Success rate':>13} {'No-answer':>10}")
print("-" * 80)
for label, n_train, sr, na in results:
    sr_str = f"{sr}%" if isinstance(sr, float) else sr
    print(f"{label:<38} {n_train:>15} {sr_str:>13} {na:>10}")